In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


  ##### <mark>**Stream Static Join implementation**</mark>

**What the output will look like**

**A new Silver/curated streaming-enriched table like:**

 **silver_order_events_enriched**
**with columns such as:**

- order_id
- customer_id
- product_id
- event_type
- event_time
- amount
- order_status
- product_name
- category
- price
- cost

###### <mark>**Step 1: Checking Schema and Values before Join**</mark>

In [2]:
# Read static product dimension table
dim_product_df = spark.table("dim_product")

# Check structure
dim_product_df.printSchema()

# View sample rows
dim_product_df.show(2, truncate=False)

StatementMeta(, 841b6e44-a2ee-4c0d-a536-2b21f15c3a90, 4, Finished, Available, Finished, False)

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- product_description: string (nullable = true)

+----------+------------+-----------+------+------+-----------------------------------------------------------------------------------------------------------------------------------------+
|product_id|product_name|category   |price |cost  |product_description                                                                                                                      |
+----------+------------+-----------+------+------+-----------------------------------------------------------------------------------------------------------------------------------------+
|1         |could       |Electronics|225.71|87.76 |Ten during wish. Apply laugh off share.\nLast family such wall serve sing. Song message build fine.                          

###### <mark>**STEP 2 — Read Streaming Source (Silver stream)**</mark>

In [3]:
# Read streaming data from silver layer
silver_stream_df = spark.readStream \
    .format("delta") \
    .table("silver_order_events_stream")

StatementMeta(, 841b6e44-a2ee-4c0d-a536-2b21f15c3a90, 5, Finished, Available, Finished, False)

###### <mark>**STEP 3 — Read Static Table (dim_product)**</mark>

In [4]:
# Static table (batch dataframe)
dim_product_df = spark.table("dim_product")

StatementMeta(, 841b6e44-a2ee-4c0d-a536-2b21f15c3a90, 6, Finished, Available, Finished, False)

###### <mark>**STEP 4 — Perform Stream-Static Join**</mark>

In [5]:
# Join streaming orders with static product data
enriched_stream_df = silver_stream_df.join(
    dim_product_df,
    on="product_id",
    how="left"   # keep all streaming data
)

StatementMeta(, 841b6e44-a2ee-4c0d-a536-2b21f15c3a90, 7, Finished, Available, Finished, False)

###### <mark>**STEP 5 — Select required columns (clean output)**</mark>

In [6]:
from pyspark.sql.functions import col

final_df = enriched_stream_df.select(
    col("order_id"),
    col("customer_id"),
    col("product_id"),
    col("product_name"),
    col("category"),
    col("price"),
    col("event_type"),
    col("event_time"),
    col("amount"),
    col("order_status")
)

StatementMeta(, 841b6e44-a2ee-4c0d-a536-2b21f15c3a90, 8, Finished, Available, Finished, False)

###### <mark>**STEP 6 — Write Streaming Output**</mark>

In [7]:
enriched_query = final_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "Files/streaming/checkpoints/silver_order_enriched") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .toTable("silver_order_events_enriched")

StatementMeta(, 841b6e44-a2ee-4c0d-a536-2b21f15c3a90, 9, Finished, Available, Finished, False)

###### <mark>**STEP 7 — Validate Output**</mark>

In [8]:
spark.sql("""
SELECT * 
FROM silver_order_events_enriched
ORDER BY order_id
""").show(truncate=False)

StatementMeta(, 841b6e44-a2ee-4c0d-a536-2b21f15c3a90, 10, Finished, Available, Finished, False)

+--------+-----------+----------+------------+-----------+------+---------------+-------------------+------+------------+
|order_id|customer_id|product_id|product_name|category   |price |event_type     |event_time         |amount|order_status|
+--------+-----------+----------+------------+-----------+------+---------------+-------------------+------+------------+
|900001  |101        |2001      |then        |Clothing   |295.65|order_placed   |2026-03-31 10:00:00|120.5 |CREATED     |
|900002  |102        |2002      |standard    |Electronics|118.55|order_placed   |2026-03-31 10:01:00|80.0  |CREATED     |
|900003  |103        |2003      |marriage    |Home       |11.18 |payment_success|2026-03-31 10:02:00|220.0 |PAID        |
|900004  |104        |2004      |message     |Home       |462.82|order_placed   |2026-03-31 10:03:00|55.75 |CREATED     |
|900005  |105        |2005      |per         |Beauty     |266.21|order_placed   |2026-03-31 10:05:00|310.25|CREATED     |
|900006  |106        |20